# Attention Mechanism from Scratch

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np

https://www.youtube.com/watch?v=viCl2T7vx64


In [3]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads # dimensionality reduction factor

        self.w_queries = nn.Linear(d_model, d_model) # queries
        self.w_keys = nn.Linear(d_model, d_model) # keys 
        self.w_values = nn.Linear(d_model, d_model) # values
        self.w_output = nn.Linear(d_model, d_model) # output


    def forward(self, x, mask=None):
        B, T, C = x.shape

        q = self.w_queries(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = self.w_keys(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = self.w_values(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        # attention function
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask==0, -1e9)

        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, v)

        return self.w_output(context), attn


In [ ]:
# Encoder layer (self-attention only)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.self_attn = SelfAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, 4*d_model), nn.ReLU(), nn.Linear(4*d_model, d_model))

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)


    def forward(self, x):
        attn_out, _ = self.self_attn(x)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x

In [ ]:
# Decoder layer (masked self-attn + encoder-decoder attn)

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.self_attn = SelfAttention(d_model, n_heads)
        self.enc_dec_attn = SelfAttention(d_model, n_heads)
        self.ffn = nn.Sequential(nn.Linear(d_model, 4*d_model), nn.ReLU(), nn.Linear(4*d_model, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
    
    def forward(self, x, enc_out, mask=None):
        # Masked self-attention
        attn_out, _ = self.self_attn(x, mask)
        x = self.norm1(x + attn_out)
        
        # Encoder-decoder attention
        attn_out, _ = self.enc_dec_attn(x, enc_out.detach())
        x = self.norm2(x + attn_out)
        
        ffn_out = self.ffn(x)
        x = self.norm3(x + ffn_out)
        return x


In [ ]:
# Attention applied between decoder layers 


In [ ]:
# Dummy example
d_model = 64
n_heads = 8
vocab_size = 100
seq_len = 5

# Dummy encoder (single layer)
encoder = EncoderLayer(d_model, n_heads)
decoder = DecoderLayer(d_model, n_heads)

# Dummy input: batch=1, seq=5, embed=64 (random embeddings for "Hello world")
src_emb = torch.rand(1, seq_len, d_model)  # Encoder input
tgt_emb = torch.rand(1, seq_len, d_model)  # Decoder input

# Causal mask for decoder self-attn
mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0).bool()

# Forward pass
enc_out = encoder(src_emb)
dec_out = decoder(tgt_emb, enc_out, mask)

print("Encoder output shape:", enc_out.shape)
print("Decoder output shape:", dec_out.shape)
print("Sample decoder output:\n", dec_out[0, :3, :3])  # First 3 tokens, first 3 dims

# Using Attention layer for seq2seq English to French translation 

In [ ]:
path = kagglehub.dataset_download("devicharith/language-translation-englishfrench")
print("Path to dataset files:", path)

In [7]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input, AdditiveAttention, Concatenate, TimeDistributed
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split

In [9]:
path = "C:/Users/108pa/.cache/kagglehub/datasets/devicharith/language-translation-englishfrench/versions/2/eng_-french.csv"
dataset = pd.read_csv(path)

In [10]:
dataset

,English words/sentences,French words/sentences
0,Hi.,Salut!
1,Run!,Cours !
2,Run!,Courez !
3,Who?,Qui ?
4,Wow!,Ça alors !
...,...,...
175616,"Top-down economics never works, said Obama. ""T...","« L'économie en partant du haut vers le bas, ç..."
175617,A carbon footprint is the amount of carbon dio...,Une empreinte carbone est la somme de pollutio...
175618,Death is something that we're often discourage...,La mort est une chose qu'on nous décourage sou...
175619,Since there are usually multiple websites on a...,Puisqu'il y a de multiples sites web sur chaqu...


In [12]:
dataset.duplicated().sum()

0

In [13]:
dataset.isnull().sum()

English words/sentences    0
French words/sentences     0
dtype: int64

In [14]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175621 entries, 0 to 175620
Data columns (total 2 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   English words/sentences  175621 non-null  object
 1   French words/sentences   175621 non-null  object
dtypes: object(2)
memory usage: 2.7+ MB


In [15]:
dataset.describe()

,English words/sentences,French words/sentences
count,175621,175621
unique,123100,165975
top,I can't tell you how happy I am that you've co...,Merci bien.
freq,32,9


In [16]:
# suffle the data
shuffled_df = dataset.sample(frac=1).reset_index(drop=True)

In [17]:
#split the data and convert it to array
english=np.array(shuffled_df['English words/sentences'])
french=np.array(shuffled_df['French words/sentences'])

In [20]:
len(english), english

(175621,
 array(["He's scared of dogs.", 'Are there any books under the desk?',
        'His mother prevented him from going out because she was anxious about his health.',
        ..., 'I have a pretty dog.', 'I love being a teacher.',
        'He lacks confidence.'], dtype=object))

In [23]:
len(french), french

(175621,
 array(['Il a peur des chiens.',
        'Y a-t-il quelque livre sous le bureau\u202f?',
        "Sa mère l'empêchait de sortir car elle s'inquiétait pour sa santé.",
        ..., "J'ai un beau chien.", "J'adore être enseignante.",
        'Il manque de confiance.'], dtype=object))

In [24]:
# sos and eos for decoder
french = ['sos ' + sentence + ' eos' for sentence in french]

In [25]:
french[:5]

['sos Il a peur des chiens. eos',
 'sos Y a-t-il quelque livre sous le bureau\u202f? eos',
 "sos Sa mère l'empêchait de sortir car elle s'inquiétait pour sa santé. eos",
 'sos Elle est plus âgée que Tom. eos',
 "sos Comment as-tu l'effronterie de me dire ça ? eos"]

In [26]:
# split data into 80% train 20% val
eng_train, eng_val, fre_train, fre_val = train_test_split(english, french, test_size=0.2,random_state=42)

In [27]:
print(len(eng_train))
print(len(eng_val))

140496
35125


In [39]:
# tokenize training data
eng_tokenizer = Tokenizer(filters='') # keep all tokens including special characters
eng_tokenizer.fit_on_texts(eng_train)

# update sequences with tokenized texts
eng_train_seq = eng_tokenizer.texts_to_sequences(eng_train) 
eng_val_seq = eng_tokenizer.texts_to_sequences(eng_val)

In [40]:
eng_train_seq

[[37, 29, 404, 3, 25, 98],
 [209, 899],
 [2, 47, 107, 57],
 [4, 445, 7873, 661, 349],
 [227, 2940, 3703, 7874],
 [71, 14, 5, 126, 32, 131],
 [137, 126, 50, 21, 2642],
 [28, 615, 1157],
 [4, 828, 9, 567, 62, 486, 400],
 [329, 95, 3, 201, 111, 1158],
 [8, 3541, 63],
 [19, 522, 20, 150, 553, 172],
 [6, 69, 108, 1229, 9, 17, 7875, 21, 14872],
 [8, 287, 4, 197],
 [16, 2405, 6, 4, 331, 390, 9, 41, 1643],
 [42, 90, 1450, 3, 222, 796],
 [1, 115, 12, 2, 86, 1053],
 [19, 251, 32, 34],
 [8, 6, 4347, 54, 73, 474, 422],
 [4, 198, 84, 25, 7876, 3, 326, 11, 1192],
 [7, 6, 5, 437, 764, 18, 36, 777],
 [39, 2, 509, 11, 268],
 [8, 6, 146, 3, 14, 900],
 [222, 562, 5743, 354],
 [7, 269, 36, 2406],
 [475, 21, 844, 172],
 [7, 516, 2234, 2643, 85, 4, 2564, 4918],
 [28, 2123, 65, 4, 198, 12, 28, 6987, 117],
 [83, 95, 54, 118, 198, 54, 8, 4621],
 [5, 3032, 679, 4, 554],
 [22, 326, 21, 3704, 3277, 42, 11171],
 [60, 6, 253, 224, 38, 481],
 [1, 27, 16, 1014, 2486],
 [8, 6287],
 [7, 287, 3, 31, 96, 50, 549, 2124],


In [31]:
fre_tokenizer = Tokenizer(filters='')
fre_tokenizer.fit_on_texts(fre_train)
fre_train_seq = fre_tokenizer.texts_to_sequences(fre_train)
fre_val_seq = fre_tokenizer.texts_to_sequences(fre_val)

In [32]:
fre_train_seq[:5]

[[1, 11, 239, 5, 1106, 11, 288, 302, 2],
 [1, 14, 15, 21, 385, 2],
 [1, 18, 8, 63, 5, 1130, 2],
 [1, 9, 488, 16076, 9, 1712, 2],
 [1, 139, 9, 3025, 3862, 33, 10636, 2]]

In [41]:
# padding

max_eng_length = max(len(seq) for seq in eng_train_seq)
max_fre_length = max(len(seq) for seq in fre_train_seq)

In [42]:
eng_train_pad = pad_sequences(eng_train_seq, maxlen=max_eng_length, padding='post')
eng_val_pad = pad_sequences(eng_val_seq, maxlen=max_eng_length, padding='post')
fre_train_pad = pad_sequences(fre_train_seq, maxlen=max_fre_length, padding='post')
fre_val_pad = pad_sequences(fre_val_seq, maxlen=max_fre_length, padding='post')

In [44]:
fre_train_pad[:1]

array([[   1,   11,  239,    5, 1106,   11,  288,  302,    2,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0]])

In [46]:
# len + 1 for padding
eng_vocab_size = len(eng_tokenizer.word_index) + 1
fre_vocab_size = len(fre_tokenizer.word_index) + 1
embedding_dim = 256
lstm_units = 256

In [49]:
# building model 

# encoder 
encoder_inputs = Input(shape=(max_eng_length,)) # input layer
enc_emb = Embedding(eng_vocab_size, embedding_dim)(encoder_inputs) # embedding layer
encoder_lstm = LSTM(lstm_units, return_sequences=True, return_state=True) # all the LSTM layers
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb) # encoder outputs

In [ ]:
# decoder 
decoder_inputs = Input(shape=(max_fre_length - 1, ))
dec_emb = Embedding(fre_vocab_size, embedding_dim)(decoder_inputs)

In [51]:
# attention layer
attention = AdditiveAttention()

In [52]:
# decoder lstm 
decoder_lstm = LSTM(lstm_units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

In [53]:
# attention mechanism
context_vector = attention([decoder_outputs, encoder_outputs])
attention_vector = Concatenate()([context_vector, decoder_outputs])

In [54]:
# output layer
decoder_dense = TimeDistributed(Dense(fre_vocab_size, activation='softmax'))
decoder_outputs = decoder_dense(attention_vector)

In [55]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [56]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [ ]:
batch_size = 64
epochs = 10

decoder_input_data = fre_train_pad[:, :-1] # everything excluding <eos>
decoder_target_data = fre_train_pad[:, 1:] # everything excluding <sos>

model.fit([eng_train_pad, decoder_input_data],
          decoder_target_data[:, :, np.newaxis],
          batch_size=batch_size,
          epochs=epochs,
          validation_data=([eng_val_pad, fre_val_pad[:, :-1]],
                           fre_val_pad[:, 1:, np.newaxis]))

Epoch 1/10
 536/2196 [======>.......................] - ETA: 5:01:15 - loss: 1.1459 - accuracy: 0.8698

In [ ]:
model.save('attention_translator.h5')
